In [59]:
import numpy as np 
import os
from model_evaluation_helpers import compute_ranking_metrics
from scipy.special import expit

In [60]:
def load_models(path): 
    filenames = ["1", "tenth", "half", "double", "five"]
    outputs = {filename: np.load(os.path.join(path, f"y_pred_{filename}.npy")) for filename in filenames}
    return outputs

In [ ]:
model_output_dir = os.path.join(os.path.expanduser("~"), "Downloads")
models = ["Convnext", "VT", "Efficientnet"]
outputs = {"Convnext": os.path.join(model_output_dir, "ConvNext"),
        "VT": os.path.join(model_output_dir, "VisionTransformer"), 
        "Efficientnet": os.path.join(model_output_dir, "EfficientNet")}
outputs = {key: load_models(value) for key, value in outputs.items()}

BEFORE
{'Convnext': {'1': array([[0.03726, 0.03455, 0.02965, 0.02615, 0.02959],
       [0.05823, 0.03574, 0.2673 , 0.1273 , 0.02681],
       [0.03928, 0.07104, 0.8066 , 0.3386 , 0.02443],
       ...,
       [0.0631 , 0.04147, 0.959  , 0.04178, 0.02452],
       [0.055  , 0.783  , 0.06805, 0.04327, 0.02452],
       [0.947  , 0.9326 , 0.0305 , 0.0341 , 0.02666]],
      shape=(3002, 5), dtype=float16), 'tenth': array([[0.0478 , 0.06573, 0.1418 , 0.06305, 0.0262 ],
       [0.02827, 0.04904, 0.3267 , 0.12494, 0.0237 ],
       [0.02971, 0.05225, 0.2593 , 0.3994 , 0.02089],
       ...,
       [0.27   , 0.03824, 0.835  , 0.1676 , 0.02692],
       [0.04858, 0.3535 , 0.06573, 0.0998 , 0.02307],
       [0.9287 , 0.935  , 0.1093 , 0.06085, 0.03056]],
      shape=(3002, 5), dtype=float16), 'half': array([[0.0384 , 0.0263 , 0.044  , 0.03096, 0.02806],
       [0.047  , 0.0263 , 0.121  , 0.1186 , 0.02356],
       [0.03079, 0.0851 , 0.806  , 0.234  , 0.02254],
       ...,
       [0.0645 , 0.03314, 0.974

In [62]:
print("AFTER")
print(outputs)

AFTER
{'Convnext': {'1': array([[0.50931441, 0.50863562, 0.50741141, 0.50653802, 0.50739616],
       [0.51455277, 0.50893307, 0.56643829, 0.53178691, 0.50670202],
       [0.50981777, 0.51775376, 0.69139318, 0.58385601, 0.50610703],
       ...,
       [0.51577235, 0.51036686, 0.72291841, 0.51044312, 0.50612991],
       [0.51374471, 0.68637005, 0.51700699, 0.51081679, 0.50612991],
       [0.72046661, 0.71760595, 0.50762499, 0.50852121, 0.50666388]],
      shape=(3002, 5)), 'tenth': array([[0.51194536, 0.5164278 , 0.53540209, 0.51575711, 0.50654946],
       [0.50706816, 0.51225798, 0.58094652, 0.53119417, 0.50592395],
       [0.50742667, 0.51305855, 0.56445864, 0.59854687, 0.50522213],
       ...,
       [0.5670977 , 0.50955847, 0.69740287, 0.54180282, 0.50672872],
       [0.51214361, 0.58746985, 0.5164278 , 0.52492744, 0.50576757],
       [0.71681369, 0.71810044, 0.52730131, 0.51520832, 0.50764024]],
      shape=(3002, 5)), 'half': array([[0.5095966 , 0.50657616, 0.51099981, 0.5077394 , 

In [26]:
# all data is now saved in outputs, we just need to evaluate
y_true = np.load(os.path.join(model_output_dir, "y_true.npy"))

In [27]:
# now run model evaluation
evaluated_models = {}
for model in models: 
    model_output = outputs[model]
    results = {key: compute_ranking_metrics(y_true, item) for key, item in model_output.items()}
    evaluated_models[model] = results

In [48]:
def pick_best_model(evaluated_models, metric):
    # pick "best' architecture based on highest macro_f1 across their models 
    best_results = {}
    for model in models: 
        model_output = evaluated_models[model]
        best_curr_result = 0.0
        best_version = ""
        for key in model_output.keys(): 
            result = model_output[key][metric]
            if result > best_curr_result: 
                best_curr_result = result
                best_version = key 
        best_results[model] = {"best_version": best_version, f"best_{metric}": best_curr_result}
    return best_results

In [49]:
pick_best_model(evaluated_models, "macro_f1")

{'Convnext': {'best_version': 'five', 'best_macro_f1': 0.7759569693757209},
 'VT': {'best_version': 'half', 'best_macro_f1': 0.7561804429548763},
 'Efficientnet': {'best_version': 'five', 'best_macro_f1': 0.7635071347595275}}

In [50]:
pick_best_model(evaluated_models, "macro_auroc")

{'Convnext': {'best_version': 'five', 'best_macro_auroc': 0.9431388973526025},
 'VT': {'best_version': 'half', 'best_macro_auroc': 0.9394859953732752},
 'Efficientnet': {'best_version': 'five',
  'best_macro_auroc': 0.9391912663716434}}

In [51]:
pick_best_model(evaluated_models, "micro_auroc")

{'Convnext': {'best_version': 'five', 'best_micro_auroc': 0.9477841965190942},
 'VT': {'best_version': 'half', 'best_micro_auroc': 0.9449665970646453},
 'Efficientnet': {'best_version': 'five',
  'best_micro_auroc': 0.9448393219234297}}

In [53]:
best_hyp_f1_results = {}
for model in models: 
    model_output = evaluated_models[model]
    best_hyp_f1 = 0.0 
    best_version = ""
    for key in model_output.keys(): 
        per_label_f1 = model_output[key]["per_label_f1"]
        hyp_f1 = per_label_f1[1]
        if hyp_f1 > best_hyp_f1: 
            best_hyp_f1 = hyp_f1
            best_version = key 
    best_hyp_f1_results[model] = {"best_version": best_version, "best_hyp_f1": best_hyp_f1}

best_hyp_f1_results

{'Convnext': {'best_version': '1', 'best_hyp_f1': 0.6599241466498104},
 'VT': {'best_version': 'half', 'best_hyp_f1': 0.6329866270430906},
 'Efficientnet': {'best_version': 'half', 'best_hyp_f1': 0.6381322957198443}}

In [54]:
# best Youden's J-score

best_youdens_j_results = {}
for model in models: 
    model_output = evaluated_models[model]
    best_hyp_youdens_j = 0.0
    best_version = ""
    for key in model_output.keys(): 
        per_label_spec = model_output[key]["per_label_specificity"]
        per_label_sens = model_output[key]["per_label_recall"]
        hyp_youdens_j = per_label_sens[1] + per_label_spec[1] - 1
        if hyp_youdens_j > best_hyp_youdens_j: 
            best_hyp_youdens_j = hyp_youdens_j
            best_version = key 
    model_results = {"best_version": best_version, 
                    "best_hyp_youdens_j": best_hyp_youdens_j, 
                    "best_spec": model_output[best_version]["per_label_specificity"][1],
                    "best_sens": model_output[best_version]["per_label_recall"][1]}
    best_youdens_j_results[model] = model_results
    
best_youdens_j_results

{'Convnext': {'best_version': '1',
  'best_hyp_youdens_j': 0.6446151318491744,
  'best_spec': 0.939209726443769,
  'best_sens': 0.7054054054054054},
 'VT': {'best_version': 'tenth',
  'best_hyp_youdens_j': 0.5956296722254169,
  'best_spec': 0.9361702127659575,
  'best_sens': 0.6594594594594595},
 'Efficientnet': {'best_version': 'half',
  'best_hyp_youdens_j': 0.6059742873572662,
  'best_spec': 0.9411094224924013,
  'best_sens': 0.6648648648648648}}